# Step 2.2 Topic Modeling for All EPB Abstracts

## 0 Load Data

In [ ]:
import pandas as pd
from pathlib import Path

intput_file_path = Path("../data/02-processed-data")
intput_file_name = intput_file_path/"epb_cleaned_abs.csv"
#load data
paper_df = pd.read_csv(intput_file_name)

In [ ]:
#Documents for the inputs
docs = list(paper_df.loc[:,"abstract"].values)
print("The length of the data is", len(docs))

## 1 Topic Modeling 

In [ ]:
# --- Custom Stop Words Definition ---

from nltk.corpus import stopwords
regular_stop_words = stopwords.words('english')


# These lists filter out domain-specific noise (common academic terminology)
words_1k = ['urban', 'model', 'city', 'planning', 'study', 'data', 'spatial', 'use', 'area', 'based', 'paper', 'system']
words_other = ['result', 'ha', 'method', 'approach', 'using', 'different', 'new', 'used', 'level', 'research', 'two', 'information', 'wa', 'also', 'problem', 'effect', 'type', 'show', 'case', 'impact', 'within', 'however', 'developed', 'application', 'high', 'context', 'relationship', 'may', 'potential', 'characteristic', 'finding', 'term', 'well', 'value', 'first', 'present', 'number', 'large', 'distribution', "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "finding", "built"]
additional_words = ['provide', 'technique', 'important', 'various', 'existing', 'way', 'applied', 'understanding', 'significant', 'need', 'future', 'many', 'size', 'found', 'across', 'higher', 'function', 'identify', 'presented', 'time', 'issue', 'made', 'user', 'work', 'author', 'mean', 'map', 'article']

# Combine NLTK defaults with custom domain-specific stop words
epb_stop_words = list(set(regular_stop_words + words_1k + words_other + additional_words))

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

# Load pre-trained sentence embedding model for document vectorization
# Use MPS GPU acceleration for Apple Silicon devices
doc_embedding_model = SentenceTransformer("all-mpnet-base-v2", device="mps")

# ⭐ Precompute document embeddings (calculated once, shared across all 9 analysis groups)
# Enable progress bar to monitor encoding progress
embeddings = doc_embedding_model.encode(docs, show_progress_bar=True)

# Initialize count vectorizer with fixed hyperparameters
# Extract unigrams and bigrams, filter custom EPB stopwords
# min_df=5: only keep terms appearing in at least 5 documents
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=epb_stop_words,
    min_df=5
)

# Custom Class TF-IDF transformer, suppress frequently occurring generic words
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

### 3.1 Parameter Testing (216 combinations)

In [ ]:
import pandas as pd
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from gensim.utils import simple_preprocess

def evaluate_all(documents, topic_models, model_names, top_n=10, n_sample=3):
    """
    Generate a unified evaluation table for all topic models.
    Metrics covered: 3 types of coherence, topic diversity, outlier ratio, sampled topic word sets.
    
    Parameters
    ----------
    documents : list[str]
        Raw input text corpus used for topic modeling
    topic_models : list[BERTopic]
        List of trained BERTopic models to evaluate
    model_names : list[str]
        Unique name label for each corresponding topic model
    top_n : int, default=10
        Number of top words per topic used to calculate topic diversity
    n_sample : int, default=3
        Number of sample non-outlier topics to display in output table

    Returns
    -------
    pd.DataFrame
        Aggregated evaluation metric table for all input topic models
    """
    # Preprocess corpus once for coherence calculation (shared across all models)
    texts = [simple_preprocess(doc) for doc in documents]
    dictionary = Dictionary(texts)

    rows = []
    for name, model in zip(model_names, topic_models):
        topics = model.get_topics()

        # Collect word lists for all valid non-outlier topics (skip topic -1)
        topic_words = [
            [word for word, _ in topics[tid]]
            for tid in topics if tid != -1
        ]

        # Helper function to compute target coherence metric
        def calculate_coherence(metric_type: str) -> float:
            return CoherenceModel(
                topics=topic_words,
                texts=texts,
                dictionary=dictionary,
                coherence=metric_type
            ).get_coherence()

        # Calculate three standard coherence metrics
        coherence_cv = calculate_coherence("c_v")
        coherence_npmi = calculate_coherence("c_npmi")
        coherence_umass = calculate_coherence("u_mass")

        # Compute topic diversity: unique word count / total top words across all topics
        all_topic_top_words = [
            word
            for tid in topics if tid != -1
            for word, _ in topics[tid][:top_n]
        ]
        if all_topic_top_words:
            diversity = len(set(all_topic_top_words)) / len(all_topic_top_words)
        else:
            diversity = 0.0

        # Calculate outlier document ratio (topic -1 represents unassigned noisy documents)
        topic_info_df = model.get_topic_info()
        total_doc_count = topic_info_df["Count"].sum()
        outlier_doc_count = topic_info_df.loc[topic_info_df["Topic"] == -1, "Count"].sum()
        outlier_ratio = outlier_doc_count / total_doc_count

        # Extract sample non-outlier topics with top 5 representative words
        sample_topic_strings = []
        for topic_id in sorted(topics.keys()):
            if topic_id == -1:
                continue
            top_five_words = [word for word, _ in model.get_topic(topic_id)[:5]]
            sample_topic_strings.append(f"T{topic_id}: {', '.join(top_five_words)}")
            if len(sample_topic_strings) >= n_sample:
                break
        sample_topics_combined = " | ".join(sample_topic_strings)

        # Assemble metric row for current model
        rows.append({
            "Model": name,
            "Num Topics": len(topic_words),
            "Coherence_cv": round(coherence_cv, 3),
            "Coherence_npmi": round(coherence_npmi, 3),
            "Coherence_umass": round(coherence_umass, 3),
            "Diversity": round(diversity, 3),
            "Outlier Ratio": round(outlier_ratio, 3),
            "Sample Topics": sample_topics_combined,
        })

    # Convert collected metric rows into structured DataFrame
    return pd.DataFrame(rows)

In [ ]:
import itertools
import pandas as pd
import gc
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

# Hyperparameter search grid for UMAP dimensionality reduction
umap_grid = {
    "n_neighbors": [10, 15, 20],
    "n_components": [3, 5, 7],
    "min_dist": [0.0, 0.3],
}

# Hyperparameter search grid for HDBSCAN clustering
hdbscan_grid = {
    "min_cluster_size": [8, 10, 15],
    "min_samples": [5, 10],
    "cluster_selection_method": ["eom", "leaf"],
}

# Generate all possible parameter combinations from grids
umap_combinations = list(itertools.product(*umap_grid.values()))
hdbscan_combinations = list(itertools.product(*hdbscan_grid.values()))

# Calculate total number of hyperparameter trials
total_trials = len(umap_combinations) * len(hdbscan_combinations)
print(f"Total hyperparameter combinations to run: {total_trials}")

# Store evaluation results of all models
result_rows = []
trial_index = 0

# Iterate over every UMAP parameter set
for (n_neighbors, n_components, min_dist) in umap_combinations:
    # Iterate over every HDBSCAN parameter set
    for (min_cluster_size, min_samples, cluster_method) in hdbscan_combinations:
        trial_index += 1
        # Create unique identifier string for current parameter configuration
        config_name = (
            f"nn{n_neighbors}_nc{n_components}_md{min_dist}_"
            f"mcs{min_cluster_size}_ms{min_samples}_{cluster_method}"
        )
        print(f"[{trial_index}/{total_trials}] Running configuration: {config_name}")

        # Initialize UMAP reducer with fixed cosine distance metric
        umap_model = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            min_dist=min_dist,
            metric='cosine',
            random_state=18
        )

        # Initialize HDBSCAN clustering model
        hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method=cluster_method,
            prediction_data=True
        )

        # Assemble full BERTopic pipeline with pre-defined embedding/vectorizer modules
        bertopic_model = BERTopic(
            embedding_model=doc_embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            ctfidf_model=ctfidf_model,
            top_n_words=15,
            min_topic_size=15,
            nr_topics='auto',
            language="english",
            calculate_probabilities=False,
            verbose=False
        )

        try:
            # Train topic model with precomputed embeddings to save computation time
            bertopic_model.fit_transform(docs, embeddings=embeddings)
            # Evaluate current model and get metric dataframe
            eval_df = evaluate_all(docs, [bertopic_model], [config_name])
            result_rows.append(eval_df)
        except Exception as error:
            # Skip broken configurations and print error traceback
            print(f"   Configuration skipped due to error: {str(error)}")

        # Free memory to avoid RAM overflow during grid search
        del bertopic_model
        gc.collect()

# Merge all trial results into one single summary table
final_metric_df = pd.concat(result_rows, ignore_index=True)
print("Grid search training and evaluation completed!")

In [ ]:
# Combine all single-model evaluation DataFrames into one complete result table
results_full = pd.concat(result_rows, ignore_index=True)

# Sort all hyperparameter configurations by CV coherence in descending order
# Higher coherence score indicates better topic interpretability
results_full = results_full.sort_values(by="Coherence_cv", ascending=False).reset_index(drop=True)

# Export full grid search metrics to CSV file for offline analysis
results_full.to_csv("grid_search_216_results.csv", index=False)

# Print the sorted table to view top-performing parameter combinations directly
results_full

In [ ]:
import os

# Primary screening to filter out obviously underperforming topic model configurations
# Filter rules: reasonable topic quantity + low outlier document proportion
filtered_df = results_full[
    (results_full["Num Topics"] >= 25) &    # Keep models with at least 25 distinct topics
    (results_full["Num Topics"] <= 50) &    # Limit maximum topic count to 50
    (results_full["Outlier Ratio"] <= 0.3)  # Restrict unassigned noisy documents below 30%
]

# Re-sort filtered configurations by c_v coherence score (descending order)
filtered_df = filtered_df.sort_values(by="Coherence_cv", ascending=False).reset_index(drop=True)

# Print remaining number of qualified hyperparameter combinations
print(f"Remaining configurations after primary filtering: {len(filtered_df)}")

# Export filtered high-quality results to CSV with full Chinese character compatibility
filtered_df.to_csv("filtered_results.csv", index=False, encoding="utf-8-sig")
# Print absolute file storage path for quick access
print("Filtered result file saved at:", os.path.abspath("filtered_results.csv"))

# Preview top 25 best-performing parameter sets
filtered_df.head(25)

### 3.2 Results Tuning and Comparing

In [ ]:
import copy
import os
import pandas as pd
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Three sets of clustering hyperparameter configurations selected from grid search results
cluster_configs = {
    "main":     dict(nn=15, nc=7, md=0.0, mcs=15, ms=10),  # Primary optimal configuration
    "altA":     dict(nn=10, nc=7, md=0.0, mcs=15, ms=10),  # Alternative candidate configuration
    "baseline": dict(nn=15, nc=5, md=0.0, mcs=10, ms=10),  # Baseline comparison configuration
}

# Create output folders, skip if directories already exist
os.makedirs("results", exist_ok=True)
os.makedirs("results/topics", exist_ok=True)

print("All clustering configurations loaded successfully ✅")

In [ ]:
import copy

def run_one_config(cfg_name, params):
    """
    Train six variant BERTopic pipelines under one fixed UMAP & HDBSCAN parameter set
    Evaluate all models and return metric table, trained model list and model name list
    
    Parameters
    ----------
    cfg_name : str
        Label name of current clustering configuration (main / altA / baseline)
    params : dict
        Hyperparameter dict containing nn, nc, md, mcs, ms for UMAP & HDBSCAN
    
    Returns
    -------
    tuple(pd.DataFrame, list[BERTopic], list[str])
        Evaluation result table, list of trained topic models, corresponding model name labels
    """
    print(f"\n{'='*40}\nRunning clustering configuration: {cfg_name}\n{'='*40}")

    # Rebuild UMAP dimensionality reducer with given hyperparameters
    umap_model = UMAP(
        n_neighbors=params["nn"],
        n_components=params["nc"],
        min_dist=params["md"],
        metric='cosine',
        random_state=18
    )

    # Rebuild HDBSCAN clustering model with given hyperparameters
    hdbscan_model = HDBSCAN(
        min_cluster_size=params["mcs"],
        min_samples=params["ms"],
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    # Helper function to initialize a blank base BERTopic template
    def create_base_bertopic():
        return BERTopic(
            embedding_model=doc_embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            top_n_words=15,
            min_topic_size=15,
            nr_topics='auto',
            language="english",
            calculate_probabilities=False,
            verbose=False
        )

    # 1. Base model: default BERTopic without custom CountVectorizer / ClassTfidf
    model_base = create_base_bertopic()
    model_base.fit_transform(docs)

    # 2. Apply custom CountVectorizer AFTER initial topic generation
    vec_post = CountVectorizer(ngram_range=(1,2), stop_words=epb_stop_words, min_df=5)
    model_cv_after = copy.deepcopy(model_base)
    model_cv_after.update_topics(docs, vectorizer_model=vec_post)

    # 3. Attach custom CountVectorizer BEFORE model fitting
    model_cv_on_train = create_base_bertopic()
    model_cv_on_train.vectorizer_model = vectorizer_model
    model_cv_on_train.fit_transform(docs)

    # 4. Only enable custom ClassTfidf transformer during training
    model_tfidf_only = create_base_bertopic()
    model_tfidf_only.ctfidf_model = ctfidf_model
    model_tfidf_only.fit_transform(docs)

    # 5. Use custom TF-IDF during training + update topics with CountVectorizer post-training
    vec_post_no_min = CountVectorizer(ngram_range=(1,2), stop_words=epb_stop_words)
    model_tfidf_cv_after = copy.deepcopy(model_tfidf_only)
    model_tfidf_cv_after.update_topics(
        docs,
        vectorizer_model=vec_post_no_min,
        ctfidf_model=ctfidf_model
    )

    # 6. Enable both custom CountVectorizer and ClassTfidf simultaneously during training
    model_cv_tfidf_joint = create_base_bertopic()
    model_cv_tfidf_joint.vectorizer_model = vectorizer_model
    model_cv_tfidf_joint.ctfidf_model = ctfidf_model
    model_cv_tfidf_joint.fit_transform(docs)

    # Aggregate all six trained models and their unique identifiers
    trained_models = [
        model_base,
        model_cv_after,
        model_cv_on_train,
        model_tfidf_only,
        model_tfidf_cv_after,
        model_cv_tfidf_joint
    ]
    model_labels = [
        f"{cfg_name}_1_Base",
        f"{cfg_name}_2_CV_after",
        f"{cfg_name}_3_CV_when",
        f"{cfg_name}_4_TFIDF",
        f"{cfg_name}_5_TFIDF_CV_after",
        f"{cfg_name}_6_CV_TFIDF"
    ]

    # Run unified evaluation pipeline for all six models
    eval_result_df = evaluate_all(docs, trained_models, model_labels)

    print(f"Configuration {cfg_name} training & evaluation finished ✅")
    return eval_result_df, trained_models, model_labels

In [ ]:
# Run pipeline for the primary optimal hyperparameter configuration
main_result_df, main_model_list, main_model_labels = run_one_config("main", cluster_configs["main"])
# Print full evaluation metrics table for all 6 model variants under main config
main_result_df
# Inspect topic statistics of the target model variant (index 4)
main_target_topic_info = main_model_list[4].get_topic_info()
# Export topic count metadata to CSV file
main_target_topic_info.to_csv("27_topics.csv", index=False)

In [ ]:
# Run pipeline for alternative candidate hyperparameter configuration
altA_result_df, altA_model_list, altA_model_labels = run_one_config("altA", cluster_configs["altA"])
# Print evaluation metrics table for altA config variants
altA_result_df
# Inspect topic statistics of target model variant (index 4)
altA_target_topic_info = altA_model_list[4].get_topic_info()
# Export topic metadata for alternative model
altA_target_topic_info.to_csv("33_topics.csv", index=False)

In [ ]:
# Run pipeline for baseline comparison hyperparameter configuration
baseline_result_df, baseline_model_list, baseline_model_labels = run_one_config("baseline", cluster_configs["baseline"])
# Print evaluation metrics table for baseline config variants
baseline_result_df
# Inspect topic statistics of target model variant (index 4)
baseline_model_list[4].get_topic_info()

## 2 Topic changing over period

In [ ]:
#Create list contains all year
paper_df['year'] = paper_df['year'].astype(int)
timestamps = paper_df['year'].to_list()
len(timestamps)

In [ ]:
# Calculate topic evolution over time using the selected main configuration model
# Input raw documents and corresponding timestamp data
topics_over_time_df = main_model_list[4].topics_over_time(
    docs,
    timestamps,
    nr_bins=None,
    datetime_format=None,
    evolution_tuning=True,
    global_tuning=True
)

# Serialize time-series topic data to pickle file for later reuse
topics_over_time_df.to_pickle("models/topics_over_time_tfidf.pkl")

# Generate interactive line chart visualizing topic popularity changes across time
main_model_list[4].visualize_topics_over_time(topics_over_time_df)

# Preview the first 5 rows of time-series topic data table
topics_over_time_df.head()

# Export topic time evolution table to CSV for static analysis and reporting
topics_over_time_df.to_csv("results/topics_over_time_tfidf_model.csv", index=False)

In [ ]:
# Import custom tool function to extract full word weight distribution for each topic
from src.analysistools import __get_topic_allwords__

# Retrieve all vocabulary and corresponding frequency weights for target topics
# Input: target topic ID list and trained BERTopic model
tf_ft_topic_df = __get_topic_allwords__(main_target_topic_info["Topic"], main_model_list[4])

# Export word-frequency table of each topic to CSV for manual inspection
tf_ft_topic_df.to_csv("results/topics_tf_ft_topic.csv", index=False)

## 3 Visualize 

In [ ]:
from pathlib import Path
import pandas as pd

# Reload the original cleaned raw paper dataset
input_file_dir = Path("../data/02-processed-data")
input_file_path = input_file_dir / "epb_cleaned_abs.csv"
paper_df = pd.read_csv(input_file_path)

# Remove duplicate abstracts to keep consistent with the corpus used for model training
df_corpus = paper_df.drop_duplicates(subset=["abstract"]).reset_index(drop=True)

# Generate timestamp list from publication year column
timestamps = df_corpus["year"].tolist()

# Print length check to guarantee matching document count
print(f"✅ Total timestamps generated: {len(timestamps)}")
print(f"✅ Total topic assignments from trained model: {len(main_model_list[4].topics_)}")
print("Note: The two numbers above must be identical to avoid dimension mismatch errors!")

# Extract topic label sequence from the target trained BERTopic model
target_model = main_model_list[4]
assigned_topic_labels = target_model.topics_

# Build dataframe linking each paper's year and assigned topic ID
df_plot = pd.DataFrame({
    "Publication Year": timestamps,
    "Topic_ID": assigned_topic_labels
})
# Filter out outlier documents marked as Topic -1
df_plot = df_plot[df_plot["Topic_ID"] != -1].reset_index(drop=True)
df_plot["Topic_ID"] = df_plot["Topic_ID"].astype(int)

# Extract basic topic metadata (ID + topic name), exclude outlier topic -1
df_topic_meta = target_model.get_topic_info()[["Topic", "Name"]]
df_topic_meta = df_topic_meta[df_topic_meta["Topic"] != -1].reset_index(drop=True)
df_topic_meta["Topic"] = df_topic_meta["Topic"].astype(int)

# Get sorted topic IDs ranked by document frequency (descending)
sorted_topic_ids_by_count = df_plot["Topic_ID"].value_counts().index.tolist()

# Print dimension and preview information for verification
print(f"✅ df_plot shape (valid documents): {df_plot.shape}")
print(f"✅ df_topic_meta shape (valid topics): {df_topic_meta.shape}")
print(f"✅ Total unique valid topics: {len(sorted_topic_ids_by_count)}")
print("\nPreview of df_plot:")
print(df_plot.head())

print(f"\nTotal documents in training corpus: {len(docs)}")
print(f"Length of timestamp list: {len(timestamps)}")
print(f"Length of model topic assignment array: {len(target_model.topics_)}")
print("Top 10 topics sorted by document count:", sorted_topic_ids_by_count[:10])

# Cross-check frequency ranking with BERTopic native topic count table
topic_frequency_table = target_model.get_topic_info()
topic_frequency_table = topic_frequency_table[topic_frequency_table["Topic"] != -1].sort_values("Count", ascending=False)
print("\nTop 10 topics sorted via BERTopic built-in count statistics:", topic_frequency_table["Topic"].head(10).tolist())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.gridspec import GridSpec
import os

# ==========================================
# 0. Load and prepare raw dataset
# ==========================================
# Read complete paper dataset with category labels
df_raw = pd.read_csv("data_papers/output_final_with_type.csv")

# Remove duplicate abstracts to match corpus used for topic model training
df_corpus = df_raw.drop_duplicates(subset=["abstract"]).reset_index(drop=True)

# Extract publication year sequence as time axis labels
timestamps = df_corpus["year"].tolist()

# Select the target trained BERTopic model for visualization
target_model = main_model_list[4]

# Dimension consistency check: ensure document count matches topic assignment length
assert len(timestamps) == len(target_model.topics_), \
    "Dimension mismatch: length of timestamps does not match length of topic assignments"

print(f"✅ Total timestamp records: {len(timestamps)}")
print(f"✅ Total topic assignment outputs from model: {len(target_model.topics_)}")

# ==========================================
# 1. Construct core plotting & metadata DataFrames
# ==========================================
assigned_topic_ids = target_model.topics_

# Link each paper's publication year to its assigned topic ID
df_plot = pd.DataFrame({
    "Publication Year": timestamps,
    "Topic_ID": assigned_topic_ids
})
# Exclude outlier documents marked with Topic_ID = -1
df_plot = df_plot[df_plot["Topic_ID"] != -1].reset_index(drop=True)
df_plot["Topic_ID"] = df_plot["Topic_ID"].astype(int)

# Extract topic metadata (unique ID + readable topic name), filter out outlier topic
df_topic_meta = target_model.get_topic_info()[["Topic", "Name"]]
df_topic_meta = df_topic_meta[df_topic_meta["Topic"] != -1].reset_index(drop=True)
df_topic_meta["Topic"] = df_topic_meta["Topic"].astype(int)

# Sort topic IDs globally by total document frequency (descending)
sorted_topic_ids_global = df_plot["Topic_ID"].value_counts().index.tolist()

print(f"✅ Valid plotting dataset shape (excluding outliers): {df_plot.shape}")
print(f"✅ Topic metadata table shape: {df_topic_meta.shape}")
print(f"✅ Total unique valid topics: {len(sorted_topic_ids_global)}")

# ==========================================
# 2. Configure color palette & hatch texture mapping for topics
# ==========================================
# Sequential diverging color palette for topic differentiation
custom_colors = [
    '#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#c7eae5',
    '#fddbc7', '#f4a582', '#d6604d', '#b2182b',
]
# Four distinct hatch patterns to supplement color differentiation
hatch_patterns = ['', '/', '-', '.']
# Dark background colors for high-contrast white text labels
dark_colors = ['#2166ac', '#4393c3', '#b2182b', '#d6604d']

num_colors = len(custom_colors)
num_hatches = len(hatch_patterns)
num_total_topics = len(sorted_topic_ids_global)

# Create global lookup dict mapping each topic ID to fixed color, hatch and frequency rank
topic_style_mapping = {}
for frequency_rank, tid in enumerate(sorted_topic_ids_global):
    color_index = int(frequency_rank * (num_colors - 1) / (num_total_topics - 1))
    hatch_index = frequency_rank % num_hatches
    topic_style_mapping[tid] = {
        "color": custom_colors[color_index],
        "hatch": hatch_patterns[hatch_index],
        "rank": frequency_rank,
    }

# ==========================================
# 3. Clean numeric year data
# ==========================================
# Convert year field to integer numeric type and drop invalid missing values
df_plot["Publication Year"] = pd.to_numeric(df_plot["Publication Year"], errors="coerce")
df_plot_valid = df_plot[df_plot["Publication Year"].notna()].copy()
df_plot_valid["Publication Year"] = df_plot_valid["Publication Year"].astype(int)
df_topic_meta["Topic"] = df_topic_meta["Topic"].astype(int)

# Define output directory for generated figures
output_dir = "pic"

# ==========================================
# 4. Global plotting parameter dictionary
# ==========================================
plot_params = dict(
    bar_height         = 0.75,
    row_height         = 0.55,
    fig_width          = 22,
    legend_row_height  = 0.32,
    legend_columns     = 3,
    fontsize_y_tick    = 11,
    fontsize_x_label   = 14,
    fontsize_title     = 18,
    fontsize_bar_text  = 8,
    fontsize_legend    = 11,
    min_pct_display    = 1.5,
)

# ==========================================
# 5. Define full analysis time window
# ==========================================
year_start, year_end = 1974, 2026
time_range_label = f"All Years ({year_start}–{year_end})"

# Global matplotlib style configuration for consistent academic plotting
plt.rcdefaults()
plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "white",
    "text.color"       : "black",
    "axes.labelcolor"  : "black",
    "xtick.color"      : "black",
    "ytick.color"      : "black",
    "font.sans-serif"  : ["Arial"],
    "axes.unicode_minus": False,
    "hatch.linewidth"  : 0.6,
})

# Filter dataset to target time window
df_time_window = df_plot_valid[
    (df_plot_valid["Publication Year"] >= year_start) &
    (df_plot_valid["Publication Year"] <= year_end)
]
total_window_papers = len(df_time_window)
all_year_sequence = list(range(year_start, year_end + 1))
num_years = len(all_year_sequence)

# Collect all unique topic IDs appearing within the target time window
window_topic_set = set()
for single_year in all_year_sequence:
    df_single_year = df_time_window[df_time_window["Publication Year"] == single_year]
    topic_count_series = df_single_year["Topic_ID"].value_counts()
    topic_count_series = topic_count_series[topic_count_series.index != -1]
    for tid in topic_count_series.index:
        window_topic_set.add(tid)

# Sort window topics by global document frequency rank
sorted_window_topics = sorted(
    window_topic_set,
    key=lambda tid: topic_style_mapping[tid]["rank"]
)
num_window_topics = len(sorted_window_topics)
legend_col_count = plot_params["legend_columns"]
legend_row_count = int(np.ceil(num_window_topics / legend_col_count))

# Calculate figure vertical height based on year rows + legend panel
main_plot_height = num_years * plot_params["row_height"] + 1.5
legend_panel_height = legend_row_count * plot_params["legend_row_height"] + 0.6
total_figure_height = main_plot_height + legend_panel_height

# Initialize figure with two vertical subplots (main chart + legend panel)
fig = plt.figure(figsize=(plot_params["fig_width"], total_figure_height), facecolor="white")
grid_spec = GridSpec(nrows=2, ncols=1, figure=fig,
                     height_ratios=[main_plot_height, legend_panel_height], hspace=0.05)
ax_main = fig.add_subplot(grid_spec[0, 0])
ax_legend_panel = fig.add_subplot(grid_spec[1, 0])
ax_main.set_facecolor("white")
ax_legend_panel.set_facecolor("white")

# ==========================================
# 6. Draw main horizontal stacked percentage bar chart
# ==========================================
annual_document_counts = []

for y_axis_index, target_year in enumerate(all_year_sequence):
    df_single_year = df_time_window[df_time_window["Publication Year"] == target_year]
    total_docs_year = len(df_single_year)
    annual_document_counts.append(total_docs_year)

    # Draw empty grey placeholder bar for years with zero publications
    if total_docs_year == 0:
        ax_main.barh(
            y=y_axis_index, width=100, left=0, height=plot_params["bar_height"],
            color="#eeeeee", edgecolor="#cccccc", linewidth=0.5
        )
        continue

    # Calculate per-topic document counts for current year
    yearly_topic_counts = df_single_year["Topic_ID"].value_counts()
    yearly_topic_counts = yearly_topic_counts[yearly_topic_counts.index != -1]
    total_valid_year_docs = yearly_topic_counts.sum()

    if total_valid_year_docs == 0:
        ax_main.barh(
            y=y_axis_index, width=100, left=0, height=plot_params["bar_height"],
            color="#eeeeee", edgecolor="#cccccc", linewidth=0.5
        )
        continue

    # Convert raw counts to percentage shares
    yearly_topic_pct = (yearly_topic_counts / total_valid_year_docs * 100).round(2)
    # Sort topics by global frequency rank for consistent stacking order
    sorted_yearly_topics = sorted(
        yearly_topic_pct.items(),
        key=lambda x: topic_style_mapping[x[0]]["rank"]
    )

    left_offset = 0
    for tid, percentage in sorted_yearly_topics:
        current_style = topic_style_mapping[tid]
        # Draw single horizontal stacked segment
        ax_main.barh(
            y=y_axis_index, width=percentage, left=left_offset, height=plot_params["bar_height"],
            color=current_style["color"], hatch=current_style["hatch"],
            edgecolor="white", linewidth=0.4,
        )
        # Render topic ID text inside segment if percentage exceeds display threshold
        if percentage >= plot_params["min_pct_display"]:
            # Dynamically shrink font for narrow segments
            if percentage >= 8:
                bar_text_font = plot_params["fontsize_bar_text"]
            elif percentage >= 4:
                bar_text_font = plot_params["fontsize_bar_text"] - 1
            else:
                bar_text_font = plot_params["fontsize_bar_text"] - 2
            # Switch text color for readability against dark/light backgrounds
            text_color = "white" if current_style["color"] in dark_colors else "black"
            ax_main.text(
                x=left_offset + percentage / 2, y=y_axis_index, s=f"T{tid}",
                ha="center", va="center",
                fontsize=bar_text_font, color=text_color,
                fontweight="bold", clip_on=True
            )
        left_offset += percentage

# Configure main axis layout and styling
ax_main.set_yticks(range(num_years))
ax_main.set_yticklabels(all_year_sequence, fontsize=plot_params["fontsize_y_tick"])
ax_main.set_xlim(0, 100)
ax_main.set_ylim(-0.5, num_years - 0.5)
ax_main.set_xlabel("Percentage (%)", fontsize=plot_params["fontsize_x_label"])
ax_main.set_title(
    f"{time_range_label}    (n={total_window_papers} papers)",
    fontsize=plot_params["fontsize_title"], fontweight="bold", pad=10, loc="left"
)
# Remove redundant chart spines
ax_main.spines["top"].set_visible(False)
ax_main.spines["right"].set_visible(False)
# Light dashed horizontal grid lines
ax_main.xaxis.grid(True, linestyle="--", alpha=0.4, color="#aaaaaa")
ax_main.set_axisbelow(True)
# Flip Y axis to show earliest year at top
ax_main.invert_yaxis()
ax_main.tick_params(axis="x", labelsize=plot_params["fontsize_x_label"] - 1)

# Twin right Y-axis to display annual total document count labels
ax_twin = ax_main.twinx()
ax_twin.set_ylim(ax_main.get_ylim())
ax_twin.set_yticks(range(num_years))
ax_twin.set_yticklabels(
    [f"n={doc_count}" for doc_count in annual_document_counts],
    fontsize=plot_params["fontsize_y_tick"], color="black"
)
ax_twin.set_title("Annual Publications",
                  fontsize=plot_params["fontsize_title"], fontweight="bold",
                  color="black", pad=10, loc="right")
ax_twin.spines["top"].set_visible(False)
ax_twin.spines["left"].set_visible(False)
ax_twin.spines["bottom"].set_visible(False)
ax_twin.spines["right"].set_color("#aaaaaa")
ax_twin.tick_params(axis="y", length=0)

# ==========================================
# 7. Build multi-column topic legend panel
# ==========================================
ax_legend_panel.axis("off")
# Separator horizontal line above legend items
ax_legend_panel.plot(
    [0, 1], [1.0, 1.0], color="#cccccc", linewidth=1.0,
    transform=ax_legend_panel.transAxes, clip_on=False
)

# Fixed legend patch dimensions
patch_width = 0.018
patch_height = 0.06
column_unit_width = 1.0 / legend_col_count
row_unit_height = 1.0 / (legend_row_count + 0.5)

# Map single-character hatch codes to printable legend hatch strings
hatch_display_map = {"": "", "/": "///", "-": "---", ".": "..."}

for display_rank, tid in enumerate(sorted_window_topics):
    col_index = display_rank % legend_col_count
    row_index = display_rank // legend_col_count
    topic_style = topic_style_mapping[tid]
    # Fetch original topic name text
    topic_name_record = df_topic_meta[df_topic_meta["Topic"] == int(tid)]
    raw_topic_name = topic_name_record["Name"].values[0] if not topic_name_record.empty else ""

    # Remove numeric prefix from topic name to avoid duplicate ID display
    if "_" in raw_topic_name:
        clean_topic_label = raw_topic_name.split("_", 1)[1]
    else:
        clean_topic_label = raw_topic_name

    # Truncate overly long topic names with ellipsis
    max_label_length = 52
    if len(clean_topic_label) > max_label_length:
        clean_topic_label = clean_topic_label[:max_label_length] + "…"

    # Calculate legend patch coordinate position
    patch_x_start = col_index * column_unit_width + 0.01
    patch_y_center = 1.0 - (row_index + 0.7) * row_unit_height

    # Draw colored rounded legend patch with matching hatch pattern
    legend_patch = mpatches.FancyBboxPatch(
        (patch_x_start, patch_y_center - patch_height / 2), patch_width, patch_height,
        boxstyle="round,pad=0.002",
        facecolor=topic_style["color"], edgecolor="white", linewidth=0.5,
        transform=ax_legend_panel.transAxes,
        hatch=hatch_display_map.get(topic_style["hatch"], topic_style["hatch"]),
        clip_on=False
    )
    ax_legend_panel.add_patch(legend_patch)
    # Render topic ID + cleaned name text next to color patch
    ax_legend_panel.text(
        x=patch_x_start + patch_width + 0.008, y=patch_y_center,
        s=f"Topic {tid}  {clean_topic_label}",
        transform=ax_legend_panel.transAxes,
        va="center", ha="left",
        fontsize=plot_params["fontsize_legend"],
        color="black", linespacing=1.3
    )

print(f"✅ Plot rendering complete: {num_window_topics} distinct topics, {total_window_papers} total papers, {num_years} years covered")

# ==========================================
# 8. Export high-resolution figure to disk
# ==========================================
os.makedirs(output_dir, exist_ok=True)
output_file_name = os.path.join(output_dir, f"all_years_{year_start}_{year_end}.png")
plt.savefig(output_file_name, dpi=300, bbox_inches="tight", facecolor="white")
print(f"💾 Figure saved successfully to path: {output_file_name}")

# Pop up figure preview window
plt.show()